In [1]:
import numpy as np
import pandas as pd
from orion.utils.envs import load_env
from orion.sources import RedshiftSource
from jinja2 import Template
import lightgbm as lgb
from datetime import datetime
from mlforecast.core import TimeSeries
from mlforecast.forecast import Forecast
from window_ops.expanding import expanding_mean
from window_ops.rolling import rolling_mean, rolling_std


load_env()

True

In [2]:
def get_division_categories():
    query = """
        SELECT DISTINCT
            globalplan1,
            globalplan4
        FROM sandpit.fullprice_forecast_input_weekly
    """

    return RedshiftSource(query=query).read_csv()


In [3]:
def import_data(division, category):

    query_template = """
        SELECT
        *
        FROM sandpit.fullprice_forecast_input_weekly
        WHERE globalplan1 = '{{division}}'
            AND globalplan4 = '{{category}}'
    """

    query = Template(query_template).render(
        division=division,
        category=category
    )

    data_source = RedshiftSource(query=query)

    return data_source.read_csv()


In [4]:
def map_col_types(df:pd.DataFrame) -> pd.DataFrame:   
    for c in df.columns:
        col_type = df[c].dtype
        if col_type == 'object' or col_type.name == 'category':
            df[c] = df[c].astype('category')
    return df

In [5]:
#interpolate missing discount weeks
def interpolate_discount(df, id, timeinteger, interpolate_col):
    discount_inter = (
    df
    .set_index(timeinteger)
    .groupby(id)
    .apply(lambda x: x[interpolate_col].interpolate(method='linear').fillna(0))
    .reset_index()
    )
    
    df[interpolate_col] = discount_inter[interpolate_col]
                 
    return df

In [6]:
#add time lagged column
def lagged_col(df, lag, timeinteger, id, lag_col):
    df
    df['t_lag'] = df[timeinteger] + lag
    df1 = df.rename({lag_col:'lagged_col'}, axis=1)
    dfinal = df.merge(df1[['t_lag',id,'lagged_col']], how='inner', left_on=[timeinteger,id], right_on=['t_lag',id])
    dfinal.drop(['t_lag_x', 't_lag_y'], axis=1, inplace=True)
    
    return dfinal

In [7]:
#add rolling mean column
def rolling_mean(df, rolling_window):
    df
    df['rm'] = df.sort_values('t').groupby('id')['sold'].transform(lambda x: x.rolling(rolling_window).mean())
    
    return df

In [8]:
#transform col names, map cols to category and datetime format
def transform_cols(input_df):
    newcols = (
        input_df
        .rename(columns={'id':'unique_id', 'sold':'y'})
        .assign(ds = lambda x:pd.to_datetime(x.date_week))
        .set_index('unique_id')
        .pipe(map_col_types)
    )
    
    return newcols

In [9]:
def train_test_split(input_data, forecast_window_length):

    t_max = input_data['t'].max()

    test_data = (
        input_data
        .query("t > @t_max - @forecast_window_length")
    )

    train_data = (
        input_data
        .query("t <= @t_max - @forecast_window_length")
    )

    return train_data, test_data

In [19]:
def run_forecast(division, category, id, timeinteger, interpolate_col, forecast_window_length, n_jobs, forecast_length):
    
    raw_data = import_data(division, category)
    
    interpolated_data = interpolate_discount(raw_data, id, timeinteger, interpolate_col)
    
    input_data = transform_cols(interpolated_data)

    train_data, test_data = train_test_split(input_data, forecast_window_length)

    #model_params = {}
    fcst = Forecast(models=[lgb.LGBMRegressor(n_jobs=n_jobs)],
            freq='W-WED',
            lags=[1, 2, 3, 4],
            date_features=['month', 'day', 'dayofweek', 'quarter', 'week']
        )

    fcst.fit(train_data, 
                 id_col='index', 
                 time_col='ds', 
                 target_col='y',
        )

    preds = (
            fcst.predict(
                forecast_length,
            )
            .reset_index()
        )

    return train_data, test_data, preds

In [20]:
run_forecast('MENS', 'T SHIRTS', 'id', 't', 'discount_pct', 8, 4, 6)

(                         date_week    t    year material_number  locationid  \
 unique_id                                                                     
 710803479017~114500004  2022-05-04  162  2022.0    710803479017   114500004   
 710662580001~114500004  2022-03-30  157  2022.0    710662580001   114500004   
 710624699011~114500004  2022-06-15  168  2022.0    710624699011   114500004   
 710624699011~114500004  2022-03-09  154  2022.0    710624699011   114500004   
 710624699072~114500004  2022-04-13  159  2022.0    710624699072   114500004   
 ...                            ...  ...     ...             ...         ...   
 710836755003~126500000  2022-03-09  154  2022.0    710836755003   126500000   
 710790058003~126500000  2022-06-01  166  2022.0    710790058003   126500000   
 710671438249~126500000  2022-04-06  158  2022.0    710671438249   126500000   
 710803228024~126100000  2022-03-16  155  2022.0    710803228024   126100000   
 710803228004~126100000  2022-03-30  157